# E13 — Probe Attribution → Causal Component Localization → Sparse Conditional Steering

## [문제상황]

E11의 Q-first 실험에서는 **hidden-state probe가 K/C 상태를 읽을 수 있었고**, `utility_staged` 조건에서 실제 QA Net Gain도 관찰되었다. 그러나 E11의 gate는 K/C 라벨로 학습한 probe와 gold 기반 utility model에 의존한다.

E12에서는 이를 곧바로 handcrafted scalar 또는 head norm / attention mass / shift로 치환했으나, vector에 있던 정보를 과도하게 압축하면서 최종 gate 성능이 유지되지 않았다.

## [원인]

잘 되는 E11 probe가 **실제로 transformer 내부의 어떤 component를 보고 있는지**를 먼저 확인하지 않은 상태에서 새로운 proxy를 직접 정의했다.

특히 linear probe라면 다음을 직접 분석할 수 있다.

- probe weight가 어떤 layer의 attention / FFN update와 정렬되는가?
- attention 안에서는 특정 head에 class separation contribution이 몰리는가?
- FFN에서는 특정 intermediate neuron/subspace에 contribution이 몰리는가?
- 그 component를 지우거나 C+/C− pair 사이에서 patch했을 때 probe score와 실제 generation이 바뀌는가?

## [해결책]

이 노트북은 **E11에서 실제로 성공한 probe**를 출발점으로 다음을 수행한다.

1. E11 `L13/last` 4-state probe를 그대로 로드해 K/C 축을 해석한다.
2. 동일 E11 split에서 `K@L7-Q`, `K@L13-Last`, `C@L13-Last` binary probe를 학습한다.
3. `StandardScaler → LogisticRegression`의 weight를 **raw hidden 좌표계**로 복원한다.
4. 목표 probe의 linear functional을 residual stream의 실제 update로 분해한다.
   - 각 layer의 attention head contribution
   - 각 layer의 FFN intermediate-neuron contribution
5. `fit`에서 class separation에 가장 크게 기여하는 component를 lock한다.
6. `utility_fit`에서 top component vs matched-random component를 zero-ablation하여 causal effect를 확인한다.
7. C+/C− pair에 selected component activation patching을 수행한다.
8. selected component의 **probe-aligned signed contribution**만 합친 sparse score를 만든다.
9. E11 instruction steering direction은 유지하고, 실제 적용 강도 `ρ = ||Δh|| / ||h||`를 직접 sweep한다.
10. `policy_cal`에서 sparse gate 정책을 lock한 뒤 held-out test를 실행한다.

> 이 실험의 1차 목표는 완전 label-free가 아니다. 먼저 **E11 probe가 무엇을 보고 있었는지 mechanistically localization**하고, 그 signal이 causal한지 검증한다. Few-shot / label-free discovery는 다음 단계다.

### 핵심 가설

> **E11에서 실제로 성공한 probe의 weight와 attribution을 직접 분해하여 K/C 판단에 기여하는 attention head·FFN subspace를 찾아 causal하게 검증한 뒤, 그 probe-aligned sparse representation을 조건부 steering gate로 사용할 수 있다.**

In [ ]:
# 1. 환경 설치 및 공통 설정
import sys, subprocess, os, json, time, math, re, ast as pyast, string, unicodedata, hashlib, warnings, gc
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd

TARGET_TRANSFORMERS = '4.57.1'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==' + TARGET_TRANSFORMERS,
    'accelerate', 'gdown', 'pyarrow', 'safetensors',
    'scikit-learn==1.6.1', 'joblib'
])

import torch, transformers, sklearn, joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

print('torch', torch.__version__, 'transformers', transformers.__version__,
      'sklearn', sklearn.__version__, 'numpy', np.__version__)

CFG = {
    'SEED': 42,
    'FOLD': 0,
    'BATCH_SIZE': 4,
    'MAX_NEW_TOKENS': 192,
    'MAX_PROMPT_TOKENS': 4096,

    # sparse localization
    'TOP_HEADS': 4,
    'TOP_FFN': 32,

    # 실제 applied ||delta||/||hidden|| 비율
    'RHO_GRID': [0.025, 0.05, 0.075, 0.10, 0.15],
    'CONF_GRID': [0.55, 0.65, 0.75, 0.85],

    # 개발 정책 안전 제약
    'CAL_MAX_DAMAGE_RATE': 0.03,
    'CAL_MAX_CELL_DROP': 0.05,

    # mechanistic diagnostics
    'RUN_GENERATION_ABLATION': True,
    'RUN_PATCHING': True,
    'PATCH_QUESTIONS': 24,
    'N_RANDOM_COMPONENT_CONTROLS': 5,
}

RUN_ID = 'E13_probe_attr_sparsegate_v1'
E11_RUN_ID = 'E11_run01'
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
CELL_NAMES = ['K+C+', 'K+C-', 'K-C-', 'K-C+']
WRITE_LAYERS = tuple(range(14, 19))

SOURCE_FILES = {
    'behavior': ('1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr', 'p8s_behavior_v2.csv'),
    'klabels': ('1LMrQ3R4XWTcBbDn3kk2mR1iTITYw-KFF', 'k_relabel_v2.parquet'),
}

SYSTEM_PROMPT = (
    'You are a question answering system. '
    'Answer the question based on the given context. '
    'Output the answer string itself and nothing else: no sentence, no explanation, '
    'no punctuation, no quotes. Keep it as short as possible (a name or a term).'
)

torch.manual_seed(CFG['SEED'])
np.random.seed(CFG['SEED'])

In [ ]:
# 2. Drive 연결 + E11 산출물 / 데이터 로드
from google.colab import drive
import gdown

DRIVE = Path('/content/drive')
if not os.path.ismount(DRIVE):
    drive.mount(str(DRIVE))

P8S_ROOT = DRIVE / 'MyDrive' / 'p8s'
E11_ROOT = P8S_ROOT / E11_RUN_ID
E11_FOLD = E11_ROOT / f"fold_{CFG['FOLD']}"
E11_QFIRST = E11_FOLD / 'e11-qfirst'
E13_ROOT = P8S_ROOT / RUN_ID
E13_ROOT.mkdir(parents=True, exist_ok=True)

# 설정/코드 버전이 바뀐 상태에서 같은 RUN_ID cache를 재사용하지 않도록 방지
E13_CODE_VERSION = '2026-09-26-e13-probe-attribution-v1'
RUN_MANIFEST = {
    'code_version': E13_CODE_VERSION,
    'cfg': CFG,
    'e11_run_id': E11_RUN_ID,
    'fold': CFG['FOLD'],
    'model_id': MODEL_ID,
}
manifest_path = E13_ROOT / 'RUN_MANIFEST.json'
if manifest_path.exists():
    old_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if old_manifest != RUN_MANIFEST:
        raise RuntimeError('기존 RUN_ID와 설정/코드 버전이 다릅니다. RUN_ID를 바꾸세요.')
else:
    manifest_path.write_text(json.dumps(RUN_MANIFEST, ensure_ascii=False, indent=2), encoding='utf-8')

required = [
    E11_FOLD / 'split_manifest.csv',
    E11_QFIRST / 'detectors.joblib',
    E11_QFIRST / 'vectors.npz',
    E11_QFIRST / 'generation_test_summary.csv',
    E11_QFIRST / 'neutral',
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f'E11 artifact가 없습니다: {p}\n'
            '먼저 E11 fold-0 q-first 결과가 Drive에 있어야 합니다.'
        )

WORK = Path('/content/e13_input')
WORK.mkdir(parents=True, exist_ok=True)
PATHS = {}
for name, (fid, fname) in SOURCE_FILES.items():
    p = WORK / fname
    if not p.exists():
        tmp = p.with_suffix(p.suffix + '.part')
        ok = gdown.download(id=fid, output=str(tmp), quiet=False)
        if not ok or not tmp.exists():
            raise RuntimeError('download failed: ' + fname)
        os.replace(tmp, p)
    PATHS[name] = p


def norm_answer(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    s = ''.join(c for c in s if c not in set(string.punctuation))
    return ' '.join(x for x in s.split() if x not in {'a', 'an', 'the'})


def truncate_answer(raw):
    s = str(raw).strip().split('\n')[0].strip()
    s = re.sub(r'^(answer|the answer is)\s*:?\s*', '', s, flags=re.I)
    s = s.split('. ')[0].strip().strip('\"\'')
    return re.sub(r'[.,;:\s]+$', '', s).strip()


def gold_aliases(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        items = list(value)
    else:
        try:
            items = json.loads(str(value))
        except Exception:
            try:
                items = pyast.literal_eval(str(value))
            except Exception:
                items = [str(value)]
    if isinstance(items, str):
        items = [items]
    return [str(x) for x in items if norm_answer(x)]


FRAME = pd.read_csv(PATHS['behavior']).reset_index(drop=True)
KLAB = pd.read_parquet(PATHS['klabels'])
conv = {'K+': 1, 'K-': 0, 'K−': 0}
yy = KLAB['kside_corr'].map(conv)
if yy.isna().any():
    raise ValueError('Unknown K label')
km = pd.DataFrame({'q': KLAB.question, 'k': yy.astype(int)}).drop_duplicates('q').set_index('q').k

FRAME['k'] = FRAME.question.map(km).astype(int)
FRAME['c'] = FRAME.ctx_label.map({'C+': 1, 'C-': 0, 'C−': 0}).astype(int)
FRAME['state'] = np.where(
    FRAME.k.eq(1),
    np.where(FRAME.c.eq(1), 0, 1),
    np.where(FRAME.c.eq(1), 3, 2)
).astype(int)
FRAME['aliases'] = FRAME.gold.map(gold_aliases)
FRAME['row'] = np.arange(len(FRAME))

SPLIT = pd.read_csv(E11_FOLD / 'split_manifest.csv').sort_values('row').reset_index(drop=True)
assert len(FRAME) == len(SPLIT) == 1224
assert FRAME.question.astype(str).tolist() == SPLIT.question.astype(str).tolist()

FIT = np.flatnonzero(SPLIT.role.eq('fit'))
UF = np.flatnonzero(SPLIT.role.eq('utility_fit'))
PC = np.flatnonzero(SPLIT.role.eq('policy_cal'))
TEST = np.flatnonzero(SPLIT.role.eq('test'))

print('E13 result root:', E13_ROOT)
print('roles:', {r: int((SPLIT.role == r).sum()) for r in SPLIT.role.unique()})
display(pd.crosstab(FRAME.k, FRAME.c))

In [ ]:
# 3. E11 neutral feature bank + 실제 E11 probe 로드

def load_npz_shards(folder):
    shards = []
    for p in sorted(Path(folder).glob('batch_*.npz')):
        with np.load(p, allow_pickle=False) as z:
            shards.append({k: z[k].copy() for k in z.files})
    if not shards:
        raise FileNotFoundError('No NPZ shards in ' + str(folder))
    return {k: np.concatenate([s[k] for s in shards], axis=0) for k in shards[0]}


E11_BANK = load_npz_shards(E11_QFIRST / 'neutral')
if not np.array_equal(E11_BANK['rows'], np.arange(len(FRAME))):
    raise RuntimeError('E11 neutral bank row alignment mismatch')

DET = joblib.load(E11_QFIRST / 'detectors.joblib')
E11_CANDIDATES = DET['candidates']
E11_SELECTED = DET['selected']

if (13, 'last') not in E11_CANDIDATES:
    raise KeyError("E11 candidates에 (13,'last') probe가 없습니다.")
E11_LAST4 = E11_CANDIDATES[(13, 'last')]

print('E11 selected detector:')
for l in sorted(E11_SELECTED):
    print(' L', l, 'kind =', E11_SELECTED[l]['kind'])

E11_SUMMARY = pd.read_csv(E11_QFIRST / 'generation_test_summary.csv')
display(E11_SUMMARY[['arm', 'baseline_EM', 'steered_EM', 'Recovery', 'Damage',
                     'NetGain_count', 'gate_rate']])

print("Primary mechanistic reference = E11 candidate (L13,'last'), 4096-d residual hidden")

In [ ]:
# 4. E11과 동일 model revision 로드
from getpass import getpass
from huggingface_hub import login, model_info
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU runtime이 필요합니다.')

print('GPU:', torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

HF_TOKEN = getpass('Hugging Face Read token (hf_...): ').strip()
if not HF_TOKEN:
    raise ValueError('HF token is empty')
login(token=HF_TOKEN, add_to_git_credential=False)

rev_path = E11_ROOT / 'model_revision.json'
if rev_path.exists():
    REVISION = json.loads(rev_path.read_text())['revision']
else:
    REVISION = model_info(MODEL_ID, token=HF_TOKEN).sha

TOK = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=HF_TOKEN, use_fast=True)
TOK.padding_side = 'left'
if TOK.pad_token_id is None:
    TOK.pad_token = TOK.eos_token

MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
    device_map={'': 'cuda:0'},
    attn_implementation='sdpa',
    trust_remote_code=False,
).eval()
for p in MODEL.parameters():
    p.requires_grad_(False)

assert len(MODEL.model.layers) == 32
HIDDEN = MODEL.config.hidden_size
N_HEADS = MODEL.config.num_attention_heads
HEAD_DIM = HIDDEN // N_HEADS
INTERMEDIATE = MODEL.config.intermediate_size

print('revision:', REVISION[:12])
print('hidden:', HIDDEN, 'heads:', N_HEADS, 'head_dim:', HEAD_DIM, 'ffn:', INTERMEDIATE)

In [ ]:
# 5. Q-first prompt / batching / scoring
@dataclass(frozen=True)
class PromptInput:
    row: int
    question: str
    context: str


def safe_input(i):
    r = FRAME.iloc[int(i)]
    return PromptInput(int(i), str(r.question), str(r.ctx_text))


def render_prompt(item):
    user = 'Question: ' + item.question + '\nContext: ' + item.context + '\nAnswer:'
    text = TOK.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    base = text.rfind(user)
    if base < 0:
        raise RuntimeError('user prompt not found after chat template')

    enc = TOK(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc['input_ids']
    offsets = enc['offset_mapping']

    qstart = base + len('Question: ')
    qend = qstart + len(item.question)
    cstart = base + len('Question: ') + len(item.question) + len('\nContext: ')
    qpos = [i for i, (s, e) in enumerate(offsets) if e > qstart and s < qend and e > s]
    if not qpos:
        raise RuntimeError('question span not found')
    if offsets[qpos[-1]][1] > cstart:
        raise AssertionError('Q-first causal span crossed Context')
    return {'ids': ids, 'qidx': qpos[-1]}


def pack_rows(rows):
    device = next(MODEL.parameters()).device
    items = [safe_input(i) for i in rows]
    packs = [render_prompt(x) for x in items]
    T = max(len(x['ids']) for x in packs)
    if T > CFG['MAX_PROMPT_TOKENS']:
        raise ValueError(f'prompt too long: {T}')

    B = len(rows)
    ids = torch.full((B, T), TOK.pad_token_id, dtype=torch.long, device=device)
    mask = torch.zeros_like(ids)
    qidx = []
    for j, p in enumerate(packs):
        n = len(p['ids'])
        off = T - n
        ids[j, off:] = torch.tensor(p['ids'], device=device)
        mask[j, off:] = 1
        qidx.append(off + p['qidx'])
    pos = (mask.cumsum(-1) - 1).clamp_min(0)
    return ids, mask, pos, torch.tensor(qidx, device=device), items


def score_text(raw, row):
    pred = norm_answer(truncate_answer(raw))
    gs = [norm_answer(x) for x in row.aliases]
    ca = norm_answer(row.answer_surface)
    return {
        'em': int(any(pred == g for g in gs)),
        'follows_ctx': int(bool(ca) and ca in pred),
        'pred': truncate_answer(raw),
    }


def cell_ids(k, c):
    k = np.asarray(k, int)
    c = np.asarray(c, int)
    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)

# 6. E11 probe를 K축 / C축으로 명시적으로 분리

## [해결책]

E11의 4-state probe 자체를 분석하면서 attribution 해석을 명확하게 하기 위해 동일 split에서 다음 binary probe도 학습한다.

- **K7-Q probe**: L7 Question-position hidden → K+/K−
- **K13-L probe**: L13 Last-position hidden → K+/K−
- **C13-L probe**: L13 Last-position hidden → C+/C−

또한 E11 `L13/last` 4-state probe의 네 class weight에서 K축과 C축을 직접 구성하고 binary probe 방향과 cosine similarity를 비교한다.

중요: `StandardScaler → LogisticRegression`이므로 `coef_`를 그대로 hidden-space weight로 보면 안 된다. 아래에서는 scaler를 역반영하여 **raw hidden 좌표계 weight**를 복원한다.

In [ ]:
# 6-1. binary probes + raw hidden-space weights

def fit_binary(X, y, train_idx, C=0.1):
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000, solver='lbfgs', random_state=CFG['SEED'])
    )
    m.fit(np.asarray(X, np.float32)[train_idx], np.asarray(y, int)[train_idx])
    return m


def binary_metrics(model, X, y, idx):
    p = model.predict_proba(np.asarray(X, np.float32)[idx])[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        'n': len(idx),
        'AUROC': float(roc_auc_score(np.asarray(y)[idx], p)),
        'BAcc': float(balanced_accuracy_score(np.asarray(y)[idx], pred)),
    }


def raw_binary_weight(pipe):
    sc = pipe.named_steps['standardscaler']
    lr = pipe.named_steps['logisticregression']
    wz = lr.coef_[0].astype(np.float64)
    scale = np.where(sc.scale_ == 0, 1.0, sc.scale_)
    w = wz / scale
    b = float(lr.intercept_[0] - np.sum(wz * sc.mean_ / scale))
    return w.astype(np.float32), b


def raw_multiclass_weights(pipe):
    sc = pipe.named_steps['standardscaler']
    lr = pipe.named_steps['logisticregression']
    scale = np.where(sc.scale_ == 0, 1.0, sc.scale_)
    W = lr.coef_.astype(np.float64) / scale[None, :]
    b = lr.intercept_.astype(np.float64) - (
        lr.coef_.astype(np.float64) * sc.mean_[None, :] / scale[None, :]
    ).sum(1)
    return W.astype(np.float32), b.astype(np.float32), lr.classes_.copy()


def cosine_np(a, b):
    return float(np.dot(a, b) / max(np.linalg.norm(a) * np.linalg.norm(b), 1e-12))


K7 = fit_binary(E11_BANK['Q7'], FRAME.k.to_numpy(), FIT)
K13 = fit_binary(E11_BANK['L13'], FRAME.k.to_numpy(), FIT)
C13 = fit_binary(E11_BANK['L13'], FRAME.c.to_numpy(), FIT)

DEV_PROBE_REPORT = []
for name, model, X, y in [
    ('K7_Q', K7, E11_BANK['Q7'], FRAME.k.to_numpy()),
    ('K13_L', K13, E11_BANK['L13'], FRAME.k.to_numpy()),
    ('C13_L', C13, E11_BANK['L13'], FRAME.c.to_numpy()),
]:
    for role, idx in [('utility_fit', UF), ('policy_cal', PC)]:
        DEV_PROBE_REPORT.append({'probe': name, 'role': role, **binary_metrics(model, X, y, idx)})

DEV_PROBE_REPORT = pd.DataFrame(DEV_PROBE_REPORT)
display(DEV_PROBE_REPORT)

wK7, bK7 = raw_binary_weight(K7)
wK13, bK13 = raw_binary_weight(K13)
wC13, bC13 = raw_binary_weight(C13)

W4, b4, classes4 = raw_multiclass_weights(E11_LAST4)
row_of = {int(c): i for i, c in enumerate(classes4)}
wK4 = 0.5 * (W4[row_of[0]] + W4[row_of[1]]) - 0.5 * (W4[row_of[2]] + W4[row_of[3]])
wC4 = 0.5 * (W4[row_of[0]] + W4[row_of[3]]) - 0.5 * (W4[row_of[1]] + W4[row_of[2]])

AXIS_COMPARE = {
    'cos(E11_4state_K_axis, binary_K13)': cosine_np(wK4, wK13),
    'cos(E11_4state_C_axis, binary_C13)': cosine_np(wC4, wC13),
}
print(json.dumps(AXIS_COMPARE, indent=2))

joblib.dump(
    {
        'K7': K7, 'K13': K13, 'C13': C13,
        'raw': {'wK7': wK7, 'bK7': bK7, 'wK13': wK13, 'bK13': bK13, 'wC13': wC13, 'bC13': bC13},
        'axis_compare': AXIS_COMPARE,
    },
    E13_ROOT / 'binary_probes.joblib'
)
DEV_PROBE_REPORT.to_csv(E13_ROOT / 'dev_binary_probe_report.csv', index=False)

# 7. Probe-aligned attribution을 모든 upstream layer로 분해

## [원인 → 해결책]

L13 probe가 final residual hidden을 읽는다고 해서 **L13 내부 component만** 보면 안 된다. L13 output은 앞선 residual stream을 계속 누적한 값이므로, linear probe score는 실제 실행에서 다음처럼 분해할 수 있다.

\[
w^T h_{L} = w^T h_0 + \sum_{\ell \le L} w^T a_\ell + \sum_{\ell \le L} w^T m_\ell
\]

여기서 각 attention update는 다시 실제 head별로, MLP update는 `down_proj` 직전 FFN intermediate neuron별로 정확히 분해할 수 있다.

### Attention head

\[
\mathrm{contrib}_{\ell,h} = z_{\ell,h}^T W_{O,\ell,h}^T w
\]

### FFN neuron

\[
\mathrm{contrib}_{\ell,i} = u_{\ell,i} (W_{down,\ell}^T w)_i
\]

**component 선택은 `fit`에서만 수행한다.** 각 component가 positive/negative class의 probe score 차이를 얼마나 설명하는지 `|mean_pos - mean_neg|`로 ranking한다. `utility_fit`, `policy_cal`, `test`는 selection에 사용하지 않는다.

In [ ]:
# 7-1. probe pullback vectors 준비
AXIS_SPECS = {
    'K7':  {'target_layer': 7,  'position': 'q',    'w': wK7,  'b': bK7,  'y': FRAME.k.to_numpy()},
    'K13': {'target_layer': 13, 'position': 'last', 'w': wK13, 'b': bK13, 'y': FRAME.k.to_numpy()},
    'C13': {'target_layer': 13, 'position': 'last', 'w': wC13, 'b': bC13, 'y': FRAME.c.to_numpy()},
}

PULLBACK = {}
for axis, spec in AXIS_SPECS.items():
    w = spec['w']
    head_v = {}
    ffn_v = {}
    for l in range(spec['target_layer'] + 1):
        block = MODEL.model.layers[l]
        WO = block.self_attn.o_proj.weight.detach().float().cpu().numpy()  # [hidden, hidden]
        hv = np.zeros((N_HEADS, HEAD_DIM), dtype=np.float32)
        for h in range(N_HEADS):
            sl = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
            hv[h] = WO[:, sl].T @ w
        head_v[l] = hv

        WD = block.mlp.down_proj.weight.detach().float().cpu().numpy()  # [hidden, intermediate]
        ffn_v[l] = (WD.T @ w).astype(np.float32)
    PULLBACK[axis] = {'head_v': head_v, 'ffn_v': ffn_v}

print('pullback vectors ready')

## 7-A. 교수님 코멘트 그대로: **E11 4-state probe weight 자체**도 먼저 본다

위 binary K/C probe는 K와 C를 해석하기 쉽게 분리하기 위한 보조 분석이다. 이와 별개로, 실제 E11 `L13/last` 4-state probe에서 만든 `wK4`, `wC4`를 transformer component 좌표로 직접 pull-back하여 **weight 절댓값이 어느 layer/head/FFN에 몰리는지** 확인한다.

이 표는 activation을 보기 전의 **static sensitivity**다. 이후의 `fit` contribution attribution과 함께 보며, 두 ranking이 겹치는 component를 특히 중요하게 본다.

In [ ]:
# 실제 E11 4-state probe-derived K/C contrast의 static component sensitivity

def static_pullback_tables(w, name, target_layer=13):
    head_rows = []
    ffn_rows = []
    for l in range(target_layer + 1):
        block = MODEL.model.layers[l]
        WO = block.self_attn.o_proj.weight.detach().float().cpu().numpy()
        for h in range(N_HEADS):
            sl = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
            v = WO[:, sl].T @ w
            head_rows.append({
                'axis': name, 'layer': l, 'head': h,
                'static_pullback_norm': float(np.linalg.norm(v)),
            })

        WD = block.mlp.down_proj.weight.detach().float().cpu().numpy()
        vf = WD.T @ w
        top = np.argsort(-np.abs(vf))[:256]
        for n in top:
            ffn_rows.append({
                'axis': name, 'layer': l, 'neuron': int(n),
                'static_pullback_abs': float(abs(vf[n])),
            })

    hdf = pd.DataFrame(head_rows).sort_values('static_pullback_norm', ascending=False).reset_index(drop=True)
    fdf = pd.DataFrame(ffn_rows).sort_values('static_pullback_abs', ascending=False).reset_index(drop=True)
    return hdf, fdf

E11K_STATIC_HEAD, E11K_STATIC_FFN = static_pullback_tables(wK4, 'E11_4state_K_axis')
E11C_STATIC_HEAD, E11C_STATIC_FFN = static_pullback_tables(wC4, 'E11_4state_C_axis')

E11K_STATIC_HEAD.to_csv(E13_ROOT / 'E11_4state_K_static_head_pullback.csv', index=False)
E11C_STATIC_HEAD.to_csv(E13_ROOT / 'E11_4state_C_static_head_pullback.csv', index=False)
E11K_STATIC_FFN.to_csv(E13_ROOT / 'E11_4state_K_static_ffn_pullback_top256_perlayer.csv', index=False)
E11C_STATIC_FFN.to_csv(E13_ROOT / 'E11_4state_C_static_ffn_pullback_top256_perlayer.csv', index=False)

print('E11 4-state K-axis: top static heads')
display(E11K_STATIC_HEAD.head(12))
print('E11 4-state C-axis: top static heads')
display(E11C_STATIC_HEAD.head(12))

In [ ]:
# 7-2. fit-only attribution discovery

def choose_pos(x, qidx, mode):
    if mode == 'q':
        return x[torch.arange(len(x), device=x.device), qidx]
    if mode == 'last':
        return x[:, -1]
    raise ValueError(mode)


def empty_stats(spec):
    L = spec['target_layer'] + 1
    return {
        'head_sum0': np.zeros((L, N_HEADS), np.float64),
        'head_sum1': np.zeros((L, N_HEADS), np.float64),
        'head_sq': np.zeros((L, N_HEADS), np.float64),
        'ffn_sum0': np.zeros((L, INTERMEDIATE), np.float64),
        'ffn_sum1': np.zeros((L, INTERMEDIATE), np.float64),
        'ffn_sq': np.zeros((L, INTERMEDIATE), np.float64),
        'n0': 0,
        'n1': 0,
        'audit_abs_err': [],
    }


ATTR_STATS = {axis: empty_stats(spec) for axis, spec in AXIS_SPECS.items()}


class AttributionDiscoverySession:
    def __init__(self, rows, qidx):
        self.rows = np.asarray(rows, int)
        self.qidx = qidx
        self.handles = []
        self.running = {axis: np.zeros(len(rows), np.float64) for axis in AXIS_SPECS}
        self.base = {}

    def __enter__(self):
        # initial residual before block 0
        block0 = MODEL.model.layers[0]
        def first_pre(mod, args, kwargs):
            h = args[0] if len(args) else kwargs.get('hidden_states')
            if h is None:
                raise RuntimeError('hidden_states missing at layer0')
            for axis, spec in AXIS_SPECS.items():
                pos = choose_pos(h, self.qidx, spec['position'])
                b = pos.detach().float().cpu().numpy() @ spec['w']
                self.base[axis] = b.astype(np.float64)
                self.running[axis] += self.base[axis]
        self.handles.append(block0.register_forward_pre_hook(first_pre, with_kwargs=True))

        for l, block in enumerate(MODEL.model.layers[:14]):
            active_axes = [a for a, s in AXIS_SPECS.items() if l <= s['target_layer']]
            if not active_axes:
                continue

            def oproj_pre(mod, args, ll=l, axes=tuple(active_axes)):
                x = args[0]
                B, T, _ = x.shape
                heads = x.reshape(B, T, N_HEADS, HEAD_DIM)
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(heads, self.qidx, spec['position']).detach().float().cpu().numpy()
                    c = np.einsum('bhd,hd->bh', z, PULLBACK[axis]['head_v'][ll]).astype(np.float64)
                    yb = spec['y'][self.rows]
                    st = ATTR_STATS[axis]
                    st['head_sum0'][ll] += c[yb == 0].sum(0)
                    st['head_sum1'][ll] += c[yb == 1].sum(0)
                    st['head_sq'][ll] += (c ** 2).sum(0)
                    self.running[axis] += c.sum(1)
            self.handles.append(block.self_attn.o_proj.register_forward_pre_hook(oproj_pre))

            def down_pre(mod, args, ll=l, axes=tuple(active_axes)):
                u = args[0]
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(u, self.qidx, spec['position']).detach().float().cpu().numpy()
                    c = z * PULLBACK[axis]['ffn_v'][ll][None, :]
                    c = c.astype(np.float64)
                    yb = spec['y'][self.rows]
                    st = ATTR_STATS[axis]
                    st['ffn_sum0'][ll] += c[yb == 0].sum(0)
                    st['ffn_sum1'][ll] += c[yb == 1].sum(0)
                    st['ffn_sq'][ll] += (c ** 2).sum(0)
                    self.running[axis] += c.sum(1)
            self.handles.append(block.mlp.down_proj.register_forward_pre_hook(down_pre))

            target_axes = [a for a in active_axes if AXIS_SPECS[a]['target_layer'] == l]
            if target_axes:
                def block_out(mod, args, out, ll=l, axes=tuple(target_axes)):
                    h = out[0] if isinstance(out, tuple) else out
                    for axis in axes:
                        spec = AXIS_SPECS[axis]
                        z = choose_pos(h, self.qidx, spec['position']).detach().float().cpu().numpy()
                        actual = z @ spec['w']
                        err = np.abs(actual - self.running[axis])
                        ATTR_STATS[axis]['audit_abs_err'].extend(err.astype(float).tolist())
                    return out
                self.handles.append(block.register_forward_hook(block_out))
        return self

    def __exit__(self, *args):
        for h in self.handles:
            h.remove()
        self.handles = []


@torch.inference_mode()
def run_attribution_discovery(rows):
    rows = list(map(int, rows))
    bs = CFG['BATCH_SIZE']
    for st in range(0, len(rows), bs):
        ix = rows[st:st+bs]
        ids, mask, pos, qidx, _ = pack_rows(ix)
        for axis, spec in AXIS_SPECS.items():
            yb = spec['y'][ix]
            ATTR_STATS[axis]['n0'] += int((yb == 0).sum())
            ATTR_STATS[axis]['n1'] += int((yb == 1).sum())
        with AttributionDiscoverySession(ix, qidx):
            _ = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos,
                            use_cache=False, return_dict=True)
        if st % (20 * bs) == 0:
            print('attribution discovery', min(st+bs, len(rows)), '/', len(rows), flush=True)

run_attribution_discovery(FIT)
print('fit attribution pass complete')

In [ ]:
# 7-3. attribution table / component lock

def build_attr_tables(axis):
    spec = AXIS_SPECS[axis]
    st = ATTR_STATS[axis]
    n0, n1 = st['n0'], st['n1']
    n = n0 + n1
    target = spec['target_layer']

    head_rows = []
    ffn_rows = []
    for l in range(target + 1):
        m0 = st['head_sum0'][l] / max(n0, 1)
        m1 = st['head_sum1'][l] / max(n1, 1)
        mean = (st['head_sum0'][l] + st['head_sum1'][l]) / max(n, 1)
        var = np.maximum(st['head_sq'][l] / max(n, 1) - mean ** 2, 0)
        for h in range(N_HEADS):
            vnorm = float(np.linalg.norm(PULLBACK[axis]['head_v'][l][h]))
            head_rows.append({
                'axis': axis, 'layer': l, 'head': h,
                'mean_class0': float(m0[h]), 'mean_class1': float(m1[h]),
                'mean_diff': float(m1[h] - m0[h]),
                'abs_mean_diff': float(abs(m1[h] - m0[h])),
                'contribution_std': float(math.sqrt(var[h])),
                'static_pullback_norm': vnorm,
            })

        m0f = st['ffn_sum0'][l] / max(n0, 1)
        m1f = st['ffn_sum1'][l] / max(n1, 1)
        meanf = (st['ffn_sum0'][l] + st['ffn_sum1'][l]) / max(n, 1)
        varf = np.maximum(st['ffn_sq'][l] / max(n, 1) - meanf ** 2, 0)
        order = np.argsort(-np.abs(m1f - m0f))[:256]
        for neuron in order:
            ffn_rows.append({
                'axis': axis, 'layer': l, 'neuron': int(neuron),
                'mean_class0': float(m0f[neuron]), 'mean_class1': float(m1f[neuron]),
                'mean_diff': float(m1f[neuron] - m0f[neuron]),
                'abs_mean_diff': float(abs(m1f[neuron] - m0f[neuron])),
                'contribution_std': float(math.sqrt(varf[neuron])),
                'static_pullback_abs': float(abs(PULLBACK[axis]['ffn_v'][l][neuron])),
            })

    hdf = pd.DataFrame(head_rows).sort_values(
        ['abs_mean_diff', 'contribution_std', 'static_pullback_norm'], ascending=False
    ).reset_index(drop=True)
    fdf = pd.DataFrame(ffn_rows).sort_values(
        ['abs_mean_diff', 'contribution_std', 'static_pullback_abs'], ascending=False
    ).reset_index(drop=True)
    return hdf, fdf


ATTR_TABLES = {}
SELECTIONS = {}
rng = np.random.default_rng(CFG['SEED'] + 1300)

for axis in AXIS_SPECS:
    hdf, fdf = build_attr_tables(axis)
    ATTR_TABLES[axis] = {'heads': hdf, 'ffn': fdf}
    hdf.to_csv(E13_ROOT / f'{axis}_alllayer_head_attribution.csv', index=False)
    fdf.to_csv(E13_ROOT / f'{axis}_alllayer_ffn_top256_perlayer.csv', index=False)

    heads = [(int(r.layer), int(r.head)) for r in hdf.head(CFG['TOP_HEADS']).itertuples()]
    ffn = [(int(r.layer), int(r.neuron)) for r in fdf.head(CFG['TOP_FFN']).itertuples()]

    all_heads = [(l, h) for l in range(AXIS_SPECS[axis]['target_layer'] + 1) for h in range(N_HEADS)]
    all_ffn = [(l, n) for l in range(AXIS_SPECS[axis]['target_layer'] + 1) for n in range(INTERMEDIATE)]
    remain_h = [x for x in all_heads if x not in set(heads)]
    remain_f = [x for x in all_ffn if x not in set(ffn)]

    random_sets = []
    for _ in range(CFG['N_RANDOM_COMPONENT_CONTROLS']):
        rh_idx = rng.choice(len(remain_h), len(heads), replace=False)
        rf_idx = rng.choice(len(remain_f), len(ffn), replace=False)
        random_sets.append({
            'heads': [remain_h[int(i)] for i in rh_idx],
            'ffn': [remain_f[int(i)] for i in rf_idx],
        })

    SELECTIONS[axis] = {
        'heads': heads,
        'ffn': ffn,
        'random_sets': random_sets,
    }

ATTR_DIAG = {
    axis: {
        'fit_n0': int(ATTR_STATS[axis]['n0']),
        'fit_n1': int(ATTR_STATS[axis]['n1']),
        'decomposition_abs_error_mean': float(np.mean(ATTR_STATS[axis]['audit_abs_err'])),
        'decomposition_abs_error_max': float(np.max(ATTR_STATS[axis]['audit_abs_err'])),
        'top_heads': SELECTIONS[axis]['heads'],
        'top_ffn': SELECTIONS[axis]['ffn'],
    }
    for axis in AXIS_SPECS
}

(E13_ROOT / 'component_lock.json').write_text(json.dumps(SELECTIONS, ensure_ascii=False, indent=2), encoding='utf-8')
(E13_ROOT / 'attribution_diagnostics.json').write_text(json.dumps(ATTR_DIAG, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(ATTR_DIAG, indent=2))

In [ ]:
# 실제 E11 4-state probe static ranking과 binary attribution ranking의 overlap

def top_pairs(df, n):
    return set((int(r.layer), int(r.head)) for r in df.head(n).itertuples())

OVERLAP = {
    'K_top4_overlap': sorted(top_pairs(E11K_STATIC_HEAD, CFG['TOP_HEADS']) & set(SELECTIONS['K13']['heads'])),
    'C_top4_overlap': sorted(top_pairs(E11C_STATIC_HEAD, CFG['TOP_HEADS']) & set(SELECTIONS['C13']['heads'])),
    'K_top8_overlap': sorted(top_pairs(E11K_STATIC_HEAD, 8) & set(SELECTIONS['K13']['heads'])),
    'C_top8_overlap': sorted(top_pairs(E11C_STATIC_HEAD, 8) & set(SELECTIONS['C13']['heads'])),
}
print('E11 4-state static ↔ binary dynamic attribution overlap')
print(json.dumps(OVERLAP, indent=2))
(E13_ROOT / 'E11_binary_component_overlap.json').write_text(json.dumps(OVERLAP, indent=2), encoding='utf-8')

In [ ]:
# 7-4. Layer × Head heatmap + top FFN table
import matplotlib.pyplot as plt

for axis, spec in AXIS_SPECS.items():
    hdf = ATTR_TABLES[axis]['heads']
    mat = np.zeros((spec['target_layer'] + 1, N_HEADS), dtype=float)
    for r in hdf.itertuples():
        mat[int(r.layer), int(r.head)] = float(r.abs_mean_diff)

    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(mat, aspect='auto')
    ax.set_xlabel('Attention head')
    ax.set_ylabel('Layer')
    ax.set_title(f'{axis}: |class mean difference| of probe-aligned head contribution')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(E13_ROOT / f'{axis}_layer_head_attribution_heatmap.png', dpi=160)
    plt.show()

    display(ATTR_TABLES[axis]['heads'].head(12))
    display(ATTR_TABLES[axis]['ffn'].head(20))

# 8. Selected component contribution만 다시 추출해 sparse score 구성

## [해결책]

E12처럼 head vector를 norm 하나로 압축하지 않는다.

각 selected component의 **원래 E11 probe 방향으로의 signed contribution**을 그대로 합한다.

- Head: `z_head · (W_O_head^T w_probe)`
- FFN neuron: `u_i · (W_down^T w_probe)_i`

이 합을 `sparse_score`로 두고, `fit`에서 1차원 logistic calibrator만 학습한다. 즉 component별 방향을 새로 재학습하지 않고, **E11 probe가 이미 학습한 방향을 보존**한다.

In [ ]:
# 8-1. selected/random sparse score extraction for all rows

def pair_maps(pairs):
    d = {}
    for l, i in pairs:
        d.setdefault(int(l), []).append(int(i))
    return d


PRIMARY_MAPS = {
    axis: {'heads': pair_maps(sel['heads']), 'ffn': pair_maps(sel['ffn'])}
    for axis, sel in SELECTIONS.items()
}
RANDOM_MAPS = {
    axis: [
        {'heads': pair_maps(rs['heads']), 'ffn': pair_maps(rs['ffn'])}
        for rs in sel['random_sets']
    ]
    for axis, sel in SELECTIONS.items()
}


class SparseScoreSession:
    def __init__(self, qidx, B):
        self.qidx = qidx
        self.B = B
        self.handles = []
        self.primary = {axis: np.zeros(B, np.float64) for axis in AXIS_SPECS}
        self.random = {
            axis: [np.zeros(B, np.float64) for _ in range(CFG['N_RANDOM_COMPONENT_CONTROLS'])]
            for axis in AXIS_SPECS
        }

    def __enter__(self):
        for l, block in enumerate(MODEL.model.layers[:14]):
            active_axes = [a for a, s in AXIS_SPECS.items() if l <= s['target_layer']]
            if not active_axes:
                continue

            def oproj_pre(mod, args, ll=l, axes=tuple(active_axes)):
                x = args[0]
                B, T, _ = x.shape
                heads = x.reshape(B, T, N_HEADS, HEAD_DIM)
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(heads, self.qidx, spec['position']).detach().float().cpu().numpy()
                    idx = PRIMARY_MAPS[axis]['heads'].get(ll, [])
                    for h in idx:
                        self.primary[axis] += z[:, h, :] @ PULLBACK[axis]['head_v'][ll][h]
                    for ri, maps in enumerate(RANDOM_MAPS[axis]):
                        for h in maps['heads'].get(ll, []):
                            self.random[axis][ri] += z[:, h, :] @ PULLBACK[axis]['head_v'][ll][h]
            self.handles.append(block.self_attn.o_proj.register_forward_pre_hook(oproj_pre))

            def down_pre(mod, args, ll=l, axes=tuple(active_axes)):
                u = args[0]
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(u, self.qidx, spec['position']).detach().float().cpu().numpy()
                    idx = PRIMARY_MAPS[axis]['ffn'].get(ll, [])
                    if idx:
                        ii = np.asarray(idx, int)
                        self.primary[axis] += (z[:, ii] * PULLBACK[axis]['ffn_v'][ll][ii][None, :]).sum(1)
                    for ri, maps in enumerate(RANDOM_MAPS[axis]):
                        idx2 = maps['ffn'].get(ll, [])
                        if idx2:
                            ii2 = np.asarray(idx2, int)
                            self.random[axis][ri] += (z[:, ii2] * PULLBACK[axis]['ffn_v'][ll][ii2][None, :]).sum(1)
            self.handles.append(block.mlp.down_proj.register_forward_pre_hook(down_pre))
        return self

    def __exit__(self, *args):
        for h in self.handles:
            h.remove()
        self.handles = []


SCORE_PATH = E13_ROOT / 'sparse_scores_all_rows.npz'

@torch.inference_mode()
def extract_sparse_scores(rows=np.arange(len(FRAME))):
    if SCORE_PATH.exists():
        with np.load(SCORE_PATH, allow_pickle=False) as z:
            return {k: z[k].copy() for k in z.files}

    rows = list(map(int, rows))
    chunks = []
    for st in range(0, len(rows), CFG['BATCH_SIZE']):
        ix = rows[st:st+CFG['BATCH_SIZE']]
        ids, mask, pos, qidx, _ = pack_rows(ix)
        with SparseScoreSession(qidx, len(ix)) as ses:
            _ = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos,
                            use_cache=False, return_dict=True)
        d = {'rows': np.asarray(ix, np.int32)}
        for axis in AXIS_SPECS:
            d[f'{axis}_primary'] = ses.primary[axis].astype(np.float32)
            for ri, x in enumerate(ses.random[axis]):
                d[f'{axis}_random{ri}'] = x.astype(np.float32)
        chunks.append(d)
        if st % (20 * CFG['BATCH_SIZE']) == 0:
            print('sparse score', min(st+CFG['BATCH_SIZE'], len(rows)), '/', len(rows), flush=True)

    out = {k: np.concatenate([c[k] for c in chunks], 0) for k in chunks[0]}
    np.savez_compressed(SCORE_PATH, **out)
    return out

SPARSE = extract_sparse_scores()
assert np.array_equal(SPARSE['rows'], np.arange(len(FRAME)))
print('saved:', SCORE_PATH)

In [ ]:
# 8-2. 1D calibrator + matched random sparse controls

def fit_1d(score, y):
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=CFG['SEED'])
    )
    m.fit(np.asarray(score, np.float32)[FIT, None], np.asarray(y, int)[FIT])
    return m


SPARSE_MODELS = {}
RANDOM_MODELS = {}
for axis, spec in AXIS_SPECS.items():
    SPARSE_MODELS[axis] = fit_1d(SPARSE[f'{axis}_primary'], spec['y'])
    RANDOM_MODELS[axis] = [
        fit_1d(SPARSE[f'{axis}_random{i}'], spec['y'])
        for i in range(CFG['N_RANDOM_COMPONENT_CONTROLS'])
    ]


def score_metrics(model, score, y, idx):
    p = model.predict_proba(np.asarray(score, np.float32)[idx, None])[:, 1]
    return {
        'AUROC': float(roc_auc_score(np.asarray(y)[idx], p)),
        'BAcc': float(balanced_accuracy_score(np.asarray(y)[idx], p >= 0.5)),
    }


rows = []
for axis, spec in AXIS_SPECS.items():
    full_model = {'K7': K7, 'K13': K13, 'C13': C13}[axis]
    full_X = {'K7': E11_BANK['Q7'], 'K13': E11_BANK['L13'], 'C13': E11_BANK['L13']}[axis]
    y = spec['y']
    for role, idx in [('utility_fit', UF), ('policy_cal', PC)]:
        rows.append({'axis': axis, 'representation': 'full_hidden', 'role': role,
                     **binary_metrics(full_model, full_X, y, idx)})
        rows.append({'axis': axis, 'representation': 'probe_aligned_sparse', 'role': role,
                     'n': len(idx), **score_metrics(SPARSE_MODELS[axis], SPARSE[f'{axis}_primary'], y, idx)})

        vals = [score_metrics(RANDOM_MODELS[axis][i], SPARSE[f'{axis}_random{i}'], y, idx)
                for i in range(CFG['N_RANDOM_COMPONENT_CONTROLS'])]
        rows.append({
            'axis': axis, 'representation': 'matched_random_sparse_mean', 'role': role, 'n': len(idx),
            'AUROC': float(np.mean([v['AUROC'] for v in vals])),
            'BAcc': float(np.mean([v['BAcc'] for v in vals])),
        })

SPARSE_DEV_REPORT = pd.DataFrame(rows)
display(SPARSE_DEV_REPORT)
SPARSE_DEV_REPORT.to_csv(E13_ROOT / 'sparse_probe_dev_report.csv', index=False)
joblib.dump({'models': SPARSE_MODELS, 'selections': SELECTIONS}, E13_ROOT / 'sparse_probe_models.joblib')

# 9. Causal diagnostic — top component zero-ablation

## [문제상황]

Attribution이 높다는 것만으로는 그 component가 실제로 probe score를 만드는 causal source라고 할 수 없다.

## [해결책]

`utility_fit`에서만 다음을 비교한다.

- top attention heads 제거
- top FFN neurons 제거
- top heads + FFN 제거
- 같은 개수의 matched-random heads + FFN 제거

그리고 **full probe score / BAcc / 실제 generation behavior**가 random control보다 더 크게 변하는지 확인한다.

In [ ]:
# 9-1. multi-layer component ablator

def selection_maps(axis, random_index=None):
    sel = SELECTIONS[axis] if random_index is None else SELECTIONS[axis]['random_sets'][random_index]
    return pair_maps(sel['heads']), pair_maps(sel['ffn'])


class MultiLayerAblator:
    def __init__(self, qidx, position, head_map=None, ffn_map=None, target_layer=None, capture=True):
        self.qidx = qidx
        self.position = position
        self.head_map = head_map or {}
        self.ffn_map = ffn_map or {}
        self.target_layer = target_layer
        self.capture = capture
        self.phase = 'prefill'
        self.handles = []
        self.target_hidden = None

    def __enter__(self):
        layers = sorted(set(self.head_map) | set(self.ffn_map) | ({self.target_layer} if self.target_layer is not None else set()))
        for l in layers:
            block = MODEL.model.layers[l]
            heads = self.head_map.get(l, [])
            neurons = self.ffn_map.get(l, [])

            if heads:
                def oproj_pre(mod, args, hh=tuple(heads)):
                    if self.phase != 'prefill':
                        return
                    x = args[0]
                    B, T, _ = x.shape
                    hv = x.reshape(B, T, N_HEADS, HEAD_DIM).clone()
                    pos = self.qidx if self.position == 'q' else torch.full((B,), T-1, device=x.device, dtype=torch.long)
                    for j in range(B):
                        hv[j, int(pos[j]), list(hh), :] = 0
                    return (hv.reshape(B, T, -1),) + tuple(args[1:])
                self.handles.append(block.self_attn.o_proj.register_forward_pre_hook(oproj_pre))

            if neurons:
                def down_pre(mod, args, nn=tuple(neurons)):
                    if self.phase != 'prefill':
                        return
                    u = args[0].clone()
                    B, T, _ = u.shape
                    pos = self.qidx if self.position == 'q' else torch.full((B,), T-1, device=u.device, dtype=torch.long)
                    for j in range(B):
                        u[j, int(pos[j]), list(nn)] = 0
                    return (u,) + tuple(args[1:])
                self.handles.append(block.mlp.down_proj.register_forward_pre_hook(down_pre))

            if self.capture and l == self.target_layer:
                def block_out(mod, args, out):
                    if self.phase != 'prefill':
                        return out
                    h = out[0] if isinstance(out, tuple) else out
                    self.target_hidden = choose_pos(h, self.qidx, self.position).detach().float().cpu().numpy()
                    return out
                self.handles.append(block.register_forward_hook(block_out))
        return self

    def __exit__(self, *args):
        for h in self.handles:
            h.remove()
        self.handles = []


@torch.inference_mode()
def ablated_target_hidden(rows, axis, which):
    rows = list(map(int, rows))
    spec = AXIS_SPECS[axis]
    hm_top, fm_top = selection_maps(axis)
    hm_rand, fm_rand = selection_maps(axis, 0)

    if which == 'top_heads': hm, fm = hm_top, {}
    elif which == 'top_ffn': hm, fm = {}, fm_top
    elif which == 'top_both': hm, fm = hm_top, fm_top
    elif which == 'random_both': hm, fm = hm_rand, fm_rand
    else: raise ValueError(which)

    chunks = []
    for st in range(0, len(rows), CFG['BATCH_SIZE']):
        ix = rows[st:st+CFG['BATCH_SIZE']]
        ids, mask, pos, qidx, _ = pack_rows(ix)
        with MultiLayerAblator(
            qidx, spec['position'], hm, fm,
            target_layer=spec['target_layer'], capture=True
        ) as ses:
            _ = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos,
                            use_cache=False, return_dict=True)
        chunks.append(ses.target_hidden.astype(np.float32))
    return np.concatenate(chunks, 0)


ABL_REPORT = []
for axis, spec in AXIS_SPECS.items():
    base_H = (E11_BANK['Q7'] if axis == 'K7' else E11_BANK['L13']).astype(np.float32)[UF]
    base_score = base_H @ spec['w'] + spec['b']
    base_bacc = balanced_accuracy_score(spec['y'][UF], base_score >= 0)

    for which in ['top_heads', 'top_ffn', 'top_both', 'random_both']:
        H = ablated_target_hidden(UF, axis, which)
        s = H @ spec['w'] + spec['b']
        ABL_REPORT.append({
            'axis': axis,
            'condition': which,
            'n': len(UF),
            'baseline_BAcc': float(base_bacc),
            'ablated_BAcc': float(balanced_accuracy_score(spec['y'][UF], s >= 0)),
            'delta_BAcc': float(balanced_accuracy_score(spec['y'][UF], s >= 0) - base_bacc),
            'mean_abs_probe_score_change': float(np.mean(np.abs(s - base_score))),
            'mean_signed_probe_score_change': float(np.mean(s - base_score)),
        })

CAUSAL_PREFILL = pd.DataFrame(ABL_REPORT)
display(CAUSAL_PREFILL)
CAUSAL_PREFILL.to_csv(E13_ROOT / 'causal_prefill_ablation.csv', index=False)

In [ ]:
# 9-2. utility_fit actual generation ablation: top-both vs random-both
class NullSession:
    def __init__(self, B):
        self.phase = 'prefill'
        self.action = [None] * B
    def __enter__(self): return self
    def __exit__(self, *args): pass


@torch.inference_mode()
def generate_with_ablation(rows, axis=None, random=False):
    rows = list(map(int, rows))
    ids, mask, pos, qidx, _ = pack_rows(rows)
    B = len(rows)

    eos = {TOK.eos_token_id}
    eot = TOK.convert_tokens_to_ids('<|eot_id|>')
    if isinstance(eot, int) and eot >= 0 and eot != TOK.unk_token_id:
        eos.add(eot)
    eos.discard(None)
    eosids = torch.tensor(sorted(eos), device=ids.device)

    if axis is None:
        session = NullSession(B)
    else:
        spec = AXIS_SPECS[axis]
        hm, fm = selection_maps(axis, 0 if random else None)
        session = MultiLayerAblator(qidx, spec['position'], hm, fm, capture=False)

    pieces = []
    ended = torch.zeros(B, dtype=torch.bool, device=ids.device)
    with session:
        h = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=True, return_dict=True)
        past = h.past_key_values
        logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])
        session.phase = 'decode'

        for step in range(CFG['MAX_NEW_TOKENS']):
            nxt = logits.argmax(-1)
            nxt = torch.where(ended, torch.full_like(nxt, TOK.pad_token_id), nxt)
            pieces.append(nxt.detach().cpu())
            active = ~ended
            ended |= torch.isin(nxt, eosids) & active
            if ended.all() or step + 1 == CFG['MAX_NEW_TOKENS']:
                break
            mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=mask.device)], -1)
            dpos = (mask.sum(-1) - 1)[:, None]
            h = MODEL.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=dpos,
                            past_key_values=past, use_cache=True, return_dict=True)
            past = h.past_key_values
            logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])

    texts = TOK.batch_decode(torch.stack(pieces, 1), skip_special_tokens=True)
    rec = []
    for r, t in zip(rows, texts):
        rec.append({'row': r, 'raw': t, **score_text(t, FRAME.iloc[r])})
    return rec


def cached_ablation_generation(tag, rows, axis=None, random=False):
    folder = E13_ROOT / 'gen_ablation' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rows = list(map(int, rows))
    for st in range(0, len(rows), CFG['BATCH_SIZE']):
        p = folder / f'batch_{st:06d}.json'
        if p.exists():
            continue
        ix = rows[st:st+CFG['BATCH_SIZE']]
        p.write_text(json.dumps(generate_with_ablation(ix, axis, random), ensure_ascii=False, indent=2), encoding='utf-8')
    rec = [x for p in sorted(folder.glob('batch_*.json')) for x in json.loads(p.read_text())]
    return pd.DataFrame(rec).set_index('row', drop=False).sort_index()


if CFG['RUN_GENERATION_ABLATION']:
    GEN = {'baseline': cached_ablation_generation('baseline', UF)}
    for axis in ['K7', 'C13']:
        GEN[f'{axis}_top'] = cached_ablation_generation(f'{axis}_top', UF, axis, False)
        GEN[f'{axis}_random'] = cached_ablation_generation(f'{axis}_random', UF, axis, True)

    b = GEN['baseline'].loc[UF, 'em'].to_numpy(int)
    report = []
    for name, df in GEN.items():
        s = df.loc[UF, 'em'].to_numpy(int)
        d = s - b
        report.append({
            'condition': name,
            'EM': float(s.mean()),
            'Recovery_vs_baseline': int((d == 1).sum()),
            'Damage_vs_baseline': int((d == -1).sum()),
            'NetGain_vs_baseline': int(d.sum()),
            'follows_context_rate': float(df.loc[UF, 'follows_ctx'].mean()),
        })
    GEN_ABL_REPORT = pd.DataFrame(report)
    display(GEN_ABL_REPORT)
    GEN_ABL_REPORT.to_csv(E13_ROOT / 'causal_generation_ablation.csv', index=False)
else:
    print('RUN_GENERATION_ABLATION=False')

# 10. C+/C− paired activation patching

## [해결책]

동일 Question의 C+ / C− pair를 이용해 **selected C13 component activation 자체를 donor 조건에서 target 조건으로 복사**한다.

- C+ component → C− run
- C− component → C+ run

관찰:

- full C13 probe score가 donor 방향으로 이동하는가?
- `follows_context` behavior가 함께 바뀌는가?
- top component patch가 matched-random patch보다 큰가?

이 실험은 `utility_fit`에서만 수행하며 method selection에는 사용하지 않는다.

In [ ]:
# 10-1. paired patching
class MultiLayerPairPatch:
    def __init__(self, qidx, donor, target, head_map=None, ffn_map=None, target_layer=13):
        self.qidx = qidx
        self.donor = int(donor)
        self.target = int(target)
        self.head_map = head_map or {}
        self.ffn_map = ffn_map or {}
        self.target_layer = target_layer
        self.phase = 'prefill'
        self.handles = []
        self.target_hidden = None

    def __enter__(self):
        layers = sorted(set(self.head_map) | set(self.ffn_map) | {self.target_layer})
        for l in layers:
            block = MODEL.model.layers[l]
            heads = self.head_map.get(l, [])
            neurons = self.ffn_map.get(l, [])

            if heads:
                def oproj_pre(mod, args, hh=tuple(heads)):
                    if self.phase != 'prefill': return
                    x = args[0]
                    B, T, _ = x.shape
                    hv = x.reshape(B, T, N_HEADS, HEAD_DIM).clone()
                    hv[self.target, -1, list(hh), :] = hv[self.donor, -1, list(hh), :]
                    return (hv.reshape(B, T, -1),) + tuple(args[1:])
                self.handles.append(block.self_attn.o_proj.register_forward_pre_hook(oproj_pre))

            if neurons:
                def down_pre(mod, args, nn=tuple(neurons)):
                    if self.phase != 'prefill': return
                    u = args[0].clone()
                    u[self.target, -1, list(nn)] = u[self.donor, -1, list(nn)]
                    return (u,) + tuple(args[1:])
                self.handles.append(block.mlp.down_proj.register_forward_pre_hook(down_pre))

            if l == self.target_layer:
                def block_out(mod, args, out):
                    if self.phase == 'prefill':
                        h = out[0] if isinstance(out, tuple) else out
                        self.target_hidden = h[self.target, -1].detach().float().cpu().numpy()
                    return out
                self.handles.append(block.register_forward_hook(block_out))
        return self

    def __exit__(self, *args):
        for h in self.handles: h.remove()
        self.handles = []


@torch.inference_mode()
def run_pair_patch(rows2, donor, target, head_map=None, ffn_map=None):
    ids, mask, pos, qidx, _ = pack_rows(rows2)
    eos = {TOK.eos_token_id}
    eot = TOK.convert_tokens_to_ids('<|eot_id|>')
    if isinstance(eot, int) and eot >= 0 and eot != TOK.unk_token_id:
        eos.add(eot)
    eos.discard(None)
    eosids = torch.tensor(sorted(eos), device=ids.device)

    ses = MultiLayerPairPatch(qidx, donor, target, head_map, ffn_map, 13)
    pieces = []
    ended = torch.zeros(2, dtype=torch.bool, device=ids.device)
    with ses:
        h = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=True, return_dict=True)
        past = h.past_key_values
        logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])
        ses.phase = 'decode'
        for step in range(CFG['MAX_NEW_TOKENS']):
            nxt = logits.argmax(-1)
            nxt = torch.where(ended, torch.full_like(nxt, TOK.pad_token_id), nxt)
            pieces.append(nxt.detach().cpu())
            active = ~ended
            ended |= torch.isin(nxt, eosids) & active
            if ended.all() or step + 1 == CFG['MAX_NEW_TOKENS']:
                break
            mask = torch.cat([mask, torch.ones((2, 1), dtype=mask.dtype, device=mask.device)], -1)
            dpos = (mask.sum(-1) - 1)[:, None]
            h = MODEL.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=dpos,
                            past_key_values=past, use_cache=True, return_dict=True)
            past = h.past_key_values
            logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])

    texts = TOK.batch_decode(torch.stack(pieces, 1), skip_special_tokens=True)
    tr = rows2[target]
    return {
        'row': int(tr),
        'C_probe_score': float(ses.target_hidden @ wC13 + bC13),
        'raw': texts[target],
        **score_text(texts[target], FRAME.iloc[tr]),
    }


if CFG['RUN_PATCHING']:
    c_top_h, c_top_f = selection_maps('C13')
    c_rand_h, c_rand_f = selection_maps('C13', 0)

    pairs = []
    for q, g in FRAME.iloc[UF].groupby('question', sort=False):
        rr = g.row.to_numpy()
        if len(rr) == 2 and set(FRAME.iloc[rr].c.tolist()) == {0, 1}:
            cp = int(rr[FRAME.iloc[rr].c.to_numpy() == 1][0])
            cm = int(rr[FRAME.iloc[rr].c.to_numpy() == 0][0])
            pairs.append((q, cp, cm))
    pairs = pairs[:CFG['PATCH_QUESTIONS']]

    rows = []
    for q, cp, cm in pairs:
        pair = [cp, cm]
        # target C- : donor C+
        for direction, donor, target in [('Cplus_to_Cminus', 0, 1), ('Cminus_to_Cplus', 1, 0)]:
            base = run_pair_patch(pair, donor, target, {}, {})
            top = run_pair_patch(pair, donor, target, c_top_h, c_top_f)
            rnd = run_pair_patch(pair, donor, target, c_rand_h, c_rand_f)
            for comp, res in [('top', top), ('random', rnd)]:
                rows.append({
                    'question': q,
                    'direction': direction,
                    'component': comp,
                    'target_row': res['row'],
                    'baseline_C_score': base['C_probe_score'],
                    'patched_C_score': res['C_probe_score'],
                    'delta_C_score': res['C_probe_score'] - base['C_probe_score'],
                    'baseline_EM': base['em'],
                    'patched_EM': res['em'],
                    'baseline_follows_ctx': base['follows_ctx'],
                    'patched_follows_ctx': res['follows_ctx'],
                })

    PATCH_REPORT = pd.DataFrame(rows)
    summary = []
    for (direction, comp), g in PATCH_REPORT.groupby(['direction', 'component']):
        summary.append({
            'direction': direction,
            'component': comp,
            'n': len(g),
            'mean_delta_C_score': float(g.delta_C_score.mean()),
            'context_follow_flip_rate': float(np.mean(g.baseline_follows_ctx.to_numpy() != g.patched_follows_ctx.to_numpy())),
            'EM_delta': float((g.patched_EM - g.baseline_EM).mean()),
        })
    PATCH_SUMMARY = pd.DataFrame(summary)
    display(PATCH_SUMMARY)
    PATCH_REPORT.to_csv(E13_ROOT / 'paired_patching_rows.csv', index=False)
    PATCH_SUMMARY.to_csv(E13_ROOT / 'paired_patching_summary.csv', index=False)
else:
    print('RUN_PATCHING=False')

# 11. Steering actuator — alpha가 아니라 실제 강도 ρ를 직접 sweep

## [문제상황]

E11은 `alpha × vector`를 더한 뒤 `MAX_DELTA_RATIO=0.05`로 cap했다. 따라서 alpha를 키워도 이미 cap에 걸린 샘플에서는 **실제 intervention strength가 같아질 수 있다.**

## [해결책]

E13에서는 E11의 **instruction contrast direction**만 재사용하고, 각 write layer에서 실제 perturbation norm을 직접 고정한다.

\[
\|\Delta h\| = \rho \|h\|
\]

- `ρ = 0.025, 0.05, 0.075, 0.10, 0.15`
- retain = instruction-context 방향의 반대
- accept = instruction-context 방향
- write = L14~18

따라서 이번 sweep은 “요청한 alpha”가 아니라 **실제 hidden 대비 적용 강도**를 비교한다.

In [ ]:
# 11-1. E11 instruction direction + generation engine
with np.load(E11_QFIRST / 'vectors.npz', allow_pickle=False) as z:
    E11_VEC = {l: z[f'instruction__{l}'].astype(np.float32) for l in WRITE_LAYERS}

UNIT_DIR = {}
for l, v in E11_VEC.items():
    n = np.linalg.norm(v)
    if n < 1e-10:
        raise ValueError(f'degenerate instruction vector L{l}')
    UNIT_DIR[l] = (v / n).astype(np.float32)

rng = np.random.default_rng(CFG['SEED'] + 909)
RANDOM_UNIT_DIR = {}
for l in WRITE_LAYERS:
    v = rng.standard_normal(HIDDEN).astype(np.float32)
    RANDOM_UNIT_DIR[l] = v / np.linalg.norm(v)


class FixedSteerSession:
    def __init__(self, B, kind=None, rho=0.0, random_direction=False):
        self.kind = kind
        self.rho = float(rho)
        self.random_direction = random_direction
        self.phase = 'prefill'
        self.handles = []
        self.action = [kind if kind else None] * B
        self.max_ratio = np.zeros(B, np.float32)
        self.scores = [{} for _ in range(B)]

    def __enter__(self):
        if not self.kind:
            return self
        dirs = RANDOM_UNIT_DIR if self.random_direction else UNIT_DIR
        sign = -1.0 if self.kind == 'retain_mid' else 1.0
        for l in WRITE_LAYERS:
            block = MODEL.model.layers[l]
            def write_hook(mod, args, out, ll=l):
                h = out[0] if isinstance(out, tuple) else out
                last = h[:, -1, :]
                u = torch.as_tensor(dirs[ll], device=h.device, dtype=h.dtype)
                delta = sign * self.rho * last.float().norm(dim=-1, keepdim=True).to(h.dtype) * u[None, :]
                ratio = delta.float().norm(dim=-1) / last.float().norm(dim=-1).clamp_min(1e-12)
                self.max_ratio = np.maximum(self.max_ratio, ratio.detach().cpu().numpy())
                new = h.clone()
                new[:, -1, :] = last + delta
                return (new,) + out[1:] if isinstance(out, tuple) else new
            self.handles.append(block.register_forward_hook(write_hook))
        return self

    def __exit__(self, *args):
        for h in self.handles: h.remove()
        self.handles = []


class GateSteerSession:
    def __init__(self, qidx, policy, gate_family='sparse', random_direction=False):
        self.qidx = qidx
        self.policy = policy
        self.gate_family = gate_family
        self.random_direction = random_direction
        self.phase = 'prefill'
        self.handles = []
        B = len(qidx)
        self.action = [None] * B
        self.max_ratio = np.zeros(B, np.float32)
        self.scores = [{} for _ in range(B)]
        self.running = {axis: np.zeros(B, np.float64) for axis in AXIS_SPECS}
        self.k7_prob = None
        self.full_k7 = None

    def __enter__(self):
        # Sparse score accumulation through upstream components
        for l, block in enumerate(MODEL.model.layers[:14]):
            active_axes = [a for a, s in AXIS_SPECS.items() if l <= s['target_layer']]
            if not active_axes:
                continue

            def oproj_pre(mod, args, ll=l, axes=tuple(active_axes)):
                if self.phase != 'prefill': return
                x = args[0]
                B, T, _ = x.shape
                heads = x.reshape(B, T, N_HEADS, HEAD_DIM)
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(heads, self.qidx, spec['position']).detach().float().cpu().numpy()
                    for h in PRIMARY_MAPS[axis]['heads'].get(ll, []):
                        self.running[axis] += z[:, h, :] @ PULLBACK[axis]['head_v'][ll][h]
            self.handles.append(block.self_attn.o_proj.register_forward_pre_hook(oproj_pre))

            def down_pre(mod, args, ll=l, axes=tuple(active_axes)):
                if self.phase != 'prefill': return
                u = args[0]
                for axis in axes:
                    spec = AXIS_SPECS[axis]
                    z = choose_pos(u, self.qidx, spec['position']).detach().float().cpu().numpy()
                    idx = PRIMARY_MAPS[axis]['ffn'].get(ll, [])
                    if idx:
                        ii = np.asarray(idx, int)
                        self.running[axis] += (z[:, ii] * PULLBACK[axis]['ffn_v'][ll][ii][None, :]).sum(1)
            self.handles.append(block.mlp.down_proj.register_forward_pre_hook(down_pre))

            if l == 7:
                def l7_out(mod, args, out):
                    if self.phase != 'prefill': return out
                    h = out[0] if isinstance(out, tuple) else out
                    q = choose_pos(h, self.qidx, 'q').detach().float().cpu().numpy()
                    self.full_k7 = K7.predict_proba(q)[:, 1]
                    self.k7_prob = SPARSE_MODELS['K7'].predict_proba(self.running['K7'][:, None].astype(np.float32))[:, 1]
                    return out
                self.handles.append(block.register_forward_hook(l7_out))

            if l == 13:
                def l13_out(mod, args, out):
                    if self.phase != 'prefill': return out
                    h = out[0] if isinstance(out, tuple) else out
                    last = h[:, -1, :].detach().float().cpu().numpy()
                    if self.gate_family == 'sparse':
                        pk13 = SPARSE_MODELS['K13'].predict_proba(self.running['K13'][:, None].astype(np.float32))[:, 1]
                        pc13 = SPARSE_MODELS['C13'].predict_proba(self.running['C13'][:, None].astype(np.float32))[:, 1]
                        pk7 = self.k7_prob
                    else:
                        pk13 = K13.predict_proba(last)[:, 1]
                        pc13 = C13.predict_proba(last)[:, 1]
                        pk7 = self.full_k7
                    pk = pk7 if self.policy['mode'] == 'staged' else pk13
                    self._route(pk, pc13, pk13)
                    return out
                self.handles.append(block.register_forward_hook(l13_out))

        dirs = RANDOM_UNIT_DIR if self.random_direction else UNIT_DIR
        for l in WRITE_LAYERS:
            block = MODEL.model.layers[l]
            def write_hook(mod, args, out, ll=l):
                h = out[0] if isinstance(out, tuple) else out
                last = h[:, -1, :]
                delta = torch.zeros_like(last)
                for j, a in enumerate(self.action):
                    if a is None: continue
                    if a == 'retain_mid':
                        rho = self.policy['retain_rho']; sign = -1.0
                    else:
                        rho = self.policy['accept_rho']; sign = 1.0
                    if rho is None: continue
                    u = torch.as_tensor(dirs[ll], device=h.device, dtype=h.dtype)
                    delta[j] = sign * float(rho) * last[j].float().norm().to(h.dtype) * u
                if not torch.any(delta): return out
                ratio = delta.float().norm(dim=-1) / last.float().norm(dim=-1).clamp_min(1e-12)
                self.max_ratio = np.maximum(self.max_ratio, ratio.detach().cpu().numpy())
                new = h.clone(); new[:, -1, :] = last + delta
                return (new,) + out[1:] if isinstance(out, tuple) else new
            self.handles.append(block.register_forward_hook(write_hook))
        return self

    def _route(self, pk, pc, pk13):
        conf = float(self.policy['confidence'])
        for j in range(len(self.action)):
            self.scores[j].update(pK_gate=float(pk[j]), pK13=float(pk13[j]), pC13=float(pc[j]))
            if self.policy.get('noop', False):
                continue
            if (self.policy['retain_rho'] is not None and pk[j] >= conf and pc[j] <= 1.0-conf):
                self.action[j] = 'retain_mid'
            elif (self.policy['accept_rho'] is not None and pk[j] <= 1.0-conf and pc[j] >= conf):
                self.action[j] = 'accept_mid'

    def __exit__(self, *args):
        for h in self.handles: h.remove()
        self.handles = []


@torch.inference_mode()
def generate_batch(rows, session_factory=None):
    rows = list(map(int, rows))
    ids, mask, pos, qidx, _ = pack_rows(rows)
    B = len(rows)

    eos = {TOK.eos_token_id}
    eot = TOK.convert_tokens_to_ids('<|eot_id|>')
    if isinstance(eot, int) and eot >= 0 and eot != TOK.unk_token_id:
        eos.add(eot)
    eos.discard(None)
    eosids = torch.tensor(sorted(eos), device=ids.device)

    ses = session_factory(qidx) if session_factory else NullSession(B)
    pieces = []
    ended = torch.zeros(B, dtype=torch.bool, device=ids.device)
    lengths = torch.zeros(B, dtype=torch.long, device=ids.device)

    if ids.device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with ses:
        h = MODEL.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=True, return_dict=True)
        past = h.past_key_values
        logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])
        for step in range(CFG['MAX_NEW_TOKENS']):
            nxt = logits.argmax(-1)
            nxt = torch.where(ended, torch.full_like(nxt, TOK.pad_token_id), nxt)
            active = ~ended
            lengths += active.long()
            pieces.append(nxt.detach().cpu())
            ended |= torch.isin(nxt, eosids) & active
            if ended.all() or step + 1 == CFG['MAX_NEW_TOKENS']:
                break
            ses.phase = 'decode'
            mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=mask.device)], -1)
            dpos = (mask.sum(-1) - 1)[:, None]
            h = MODEL.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=dpos,
                            past_key_values=past, use_cache=True, return_dict=True)
            past = h.past_key_values
            logits = MODEL.lm_head(h.last_hidden_state[:, -1, :])

    if ids.device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    texts = TOK.batch_decode(torch.stack(pieces, 1), skip_special_tokens=True)

    rec = []
    for j, (r, t) in enumerate(zip(rows, texts)):
        rec.append({
            'row': r,
            'raw': t,
            'action': ses.action[j] if hasattr(ses, 'action') and ses.action[j] is not None else 'noop',
            'seconds': elapsed / B,
            'new_tokens': int(lengths[j]),
            'max_applied_delta_ratio': float(ses.max_ratio[j]) if hasattr(ses, 'max_ratio') else 0.0,
            **(ses.scores[j] if hasattr(ses, 'scores') else {}),
        })
    return rec


def cached_generation(tag, rows, session_factory=None):
    folder = E13_ROOT / 'generation' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rows = list(map(int, rows))
    for st in range(0, len(rows), CFG['BATCH_SIZE']):
        p = folder / f'batch_{st:06d}.json'
        if p.exists(): continue
        ix = rows[st:st+CFG['BATCH_SIZE']]
        p.write_text(json.dumps(generate_batch(ix, session_factory), ensure_ascii=False, indent=2), encoding='utf-8')
        if st % (20 * CFG['BATCH_SIZE']) == 0:
            print(tag, min(st+CFG['BATCH_SIZE'], len(rows)), '/', len(rows), flush=True)
    rec = [x for p in sorted(folder.glob('batch_*.json')) for x in json.loads(p.read_text())]
    return pd.DataFrame(rec).set_index('row', drop=False).sort_index()

# 12. `policy_cal`에서 gate mode + confidence + 실제 ρ 고정

두 sparse gate 후보를 비교한다.

### staged-read
- K: L7 Question-position sparse K
- C: L13 Last-position sparse C
- write: L14~18

### late-read
- K: L13 Last-position sparse K
- C: L13 Last-position sparse C
- write: L14~18

Full-hidden binary probe도 동일한 정책 search를 수행하지만 **upper-bound diagnostic**으로만 둔다.

Test는 이 단계에서 절대 사용하지 않는다.

In [ ]:
# 12-1. policy_cal baseline + forced action bank
PC_BASE = cached_generation('policy_cal_noop', PC)

FORCED = {}
for kind in ['retain_mid', 'accept_mid']:
    for rho in CFG['RHO_GRID']:
        key = f'{kind}_rho{rho:g}'
        print('forced:', key)
        FORCED[key] = cached_generation(
            'policy_cal_force_' + key,
            PC,
            session_factory=lambda qidx, k=kind, r=rho:
                FixedSteerSession(len(qidx), kind=k, rho=r, random_direction=False)
        )


def attach_scores(df, rows):
    x = df.copy()
    vals = [score_text(x.loc[r, 'raw'], FRAME.iloc[r]) for r in rows]
    x.loc[rows, 'em'] = [v['em'] for v in vals]
    x.loc[rows, 'follows_ctx'] = [v['follows_ctx'] for v in vals]
    return x


PC_BASE = attach_scores(PC_BASE, PC)
FORCED = {k: attach_scores(v, PC) for k, v in FORCED.items()}


def probs_for_family(family, mode, rows):
    rows = np.asarray(rows, int)
    if family == 'sparse':
        pk7 = SPARSE_MODELS['K7'].predict_proba(SPARSE['K7_primary'][rows, None])[:, 1]
        pk13 = SPARSE_MODELS['K13'].predict_proba(SPARSE['K13_primary'][rows, None])[:, 1]
        pc13 = SPARSE_MODELS['C13'].predict_proba(SPARSE['C13_primary'][rows, None])[:, 1]
    else:
        pk7 = K7.predict_proba(E11_BANK['Q7'][rows].astype(np.float32))[:, 1]
        pk13 = K13.predict_proba(E11_BANK['L13'][rows].astype(np.float32))[:, 1]
        pc13 = C13.predict_proba(E11_BANK['L13'][rows].astype(np.float32))[:, 1]
    return (pk7 if mode == 'staged' else pk13), pc13, pk13


def simulate_policy(rows, family, mode, conf, retain_rho, accept_rho):
    rows = np.asarray(rows, int)
    pk, pc, _ = probs_for_family(family, mode, rows)
    base = PC_BASE.loc[rows, 'em'].to_numpy(int)
    out = base.copy()
    actions = np.array(['noop'] * len(rows), dtype=object)

    retain_mask = ((pk >= conf) & (pc <= 1-conf)) if retain_rho is not None else np.zeros(len(rows), bool)
    accept_mask = ((pk <= 1-conf) & (pc >= conf)) if accept_rho is not None else np.zeros(len(rows), bool)

    if retain_rho is not None:
        key = f'retain_mid_rho{retain_rho:g}'
        out[retain_mask] = FORCED[key].loc[rows[retain_mask], 'em'].to_numpy(int)
        actions[retain_mask] = 'retain_mid'

    if accept_rho is not None:
        key = f'accept_mid_rho{accept_rho:g}'
        out[accept_mask] = FORCED[key].loc[rows[accept_mask], 'em'].to_numpy(int)
        actions[accept_mask] = 'accept_mid'

    d = out - base
    state = FRAME.iloc[rows].state.to_numpy()
    cell_gain = []
    for s in range(4):
        m = state == s
        cell_gain.append(float(d[m].mean()) if m.any() else 0.0)

    return {
        'family': family, 'mode': mode, 'confidence': conf,
        'retain_rho': retain_rho, 'accept_rho': accept_rho,
        'baseline_EM': float(base.mean()), 'steered_EM': float(out.mean()),
        'Recovery': int((d == 1).sum()), 'Damage': int((d == -1).sum()),
        'NetGain_count': int(d.sum()), 'NetGain_rate': float(d.mean()),
        'gate_rate': float(np.mean(actions != 'noop')),
        'damage_rate_all': float(np.mean(d == -1)),
        'macro_cell_gain': float(np.mean(cell_gain)),
        'min_cell_gain': float(np.min(cell_gain)),
    }


def calibrate_family(family):
    records = []
    rho_opts = [None] + list(CFG['RHO_GRID'])
    for mode in ['staged', 'late']:
        for conf in CFG['CONF_GRID']:
            for rr in rho_opts:
                for ar in rho_opts:
                    if rr is None and ar is None: continue
                    met = simulate_policy(PC, family, mode, conf, rr, ar)
                    met['feasible'] = (
                        met['damage_rate_all'] <= CFG['CAL_MAX_DAMAGE_RATE']
                        and met['min_cell_gain'] >= -CFG['CAL_MAX_CELL_DROP']
                    )
                    records.append(met)

    d = pd.DataFrame(records)
    feasible = d[d.feasible & d.NetGain_count.gt(0)].copy()
    if len(feasible) == 0:
        lock = {
            'family': family, 'mode': 'late', 'confidence': 1.0,
            'retain_rho': None, 'accept_rho': None, 'noop': True,
            'reason': 'No positive feasible policy on policy_cal',
        }
    else:
        feasible = feasible.sort_values(
            ['NetGain_rate', 'macro_cell_gain', 'Damage', 'gate_rate'],
            ascending=[False, False, True, True]
        )
        r = feasible.iloc[0]
        lock = {
            'family': family,
            'mode': str(r['mode']),
            'confidence': float(r['confidence']),
            'retain_rho': None if pd.isna(r['retain_rho']) else float(r['retain_rho']),
            'accept_rho': None if pd.isna(r['accept_rho']) else float(r['accept_rho']),
            'noop': False,
        }
    return d, lock


SPARSE_SEARCH, SPARSE_POLICY = calibrate_family('sparse')
FULL_SEARCH, FULL_POLICY = calibrate_family('full')
SPARSE_SEARCH.to_csv(E13_ROOT / 'policy_calibration_sparse.csv', index=False)
FULL_SEARCH.to_csv(E13_ROOT / 'policy_calibration_full.csv', index=False)

POLICY_LOCK = {
    'created_before_test_evaluation': True,
    'test_used_for_selection': False,
    'top_heads': CFG['TOP_HEADS'],
    'top_ffn': CFG['TOP_FFN'],
    'sparse_policy': SPARSE_POLICY,
    'full_probe_upper_bound_policy': FULL_POLICY,
    'rho_definition': 'actual ||delta|| / ||hidden||',
    'component_lock': SELECTIONS,
}
lock_path = E13_ROOT / 'POLICY_LOCK.json'
lock_path.write_text(json.dumps(POLICY_LOCK, ensure_ascii=False, indent=2), encoding='utf-8')
LOCK_HASH = hashlib.sha256(lock_path.read_bytes()).hexdigest()

print('SPARSE POLICY:', json.dumps(SPARSE_POLICY, indent=2))
print('FULL UPPER POLICY:', json.dumps(FULL_POLICY, indent=2))
print('LOCK HASH:', LOCK_HASH[:16])

# 13. 🔒 Held-out test — lock 이후 최초 최종 평가

비교:

1. **No-op**
2. **E13 sparse probe-aligned gate** — 주 방법
3. **Full hidden probe gate** — supervised upper-bound diagnostic
4. **Sparse gate + random steering direction** — route는 동일하고 actuator direction만 random

평가:

- K/C AUROC, BAcc
- 4-state BAcc / Macro-F1
- EM
- Recovery / Damage / Net Gain
- K/C cell별 ΔEM
- intervention rate
- 실제 applied `||Δh||/||h||`
- latency

In [ ]:
# 13-1. final test generation
assert hashlib.sha256(lock_path.read_bytes()).hexdigest() == LOCK_HASH

TEST_BASE = cached_generation('test_noop', TEST)
TEST_SPARSE = cached_generation(
    'test_sparse_gate', TEST,
    session_factory=lambda qidx: GateSteerSession(qidx, SPARSE_POLICY, 'sparse', False)
)
assert hashlib.sha256(lock_path.read_bytes()).hexdigest() == LOCK_HASH

TEST_FULL = cached_generation(
    'test_full_probe_gate_DIAGNOSTIC', TEST,
    session_factory=lambda qidx: GateSteerSession(qidx, FULL_POLICY, 'full', False)
)
assert hashlib.sha256(lock_path.read_bytes()).hexdigest() == LOCK_HASH

TEST_RANDOM = cached_generation(
    'test_sparse_gate_random_direction', TEST,
    session_factory=lambda qidx: GateSteerSession(qidx, SPARSE_POLICY, 'sparse', True)
)
assert hashlib.sha256(lock_path.read_bytes()).hexdigest() == LOCK_HASH

if not np.array_equal(TEST_SPARSE.loc[TEST, 'action'].to_numpy(), TEST_RANDOM.loc[TEST, 'action'].to_numpy()):
    raise RuntimeError('Random-direction control route mismatch')


def scored(df):
    x = df.copy()
    vals = [score_text(x.loc[r, 'raw'], FRAME.iloc[r]) for r in TEST]
    x.loc[TEST, 'em'] = [v['em'] for v in vals]
    x.loc[TEST, 'follows_ctx'] = [v['follows_ctx'] for v in vals]
    return x


B = scored(TEST_BASE)
S = scored(TEST_SPARSE)
F = scored(TEST_FULL)
R = scored(TEST_RANDOM)


def qa_metrics(base, active, name):
    b = base.loc[TEST, 'em'].to_numpy(int)
    s = active.loc[TEST, 'em'].to_numpy(int)
    d = s - b
    out = {
        'method': name,
        'n': len(TEST),
        'baseline_EM': float(b.mean()),
        'steered_EM': float(s.mean()),
        'Recovery': int((d == 1).sum()),
        'Damage': int((d == -1).sum()),
        'NetGain_count': int(d.sum()),
        'NetGain_rate': float(d.mean()),
        'intervention_rate': float(active.loc[TEST, 'action'].ne('noop').mean()),
        'max_applied_delta_ratio': float(active.loc[TEST, 'max_applied_delta_ratio'].max()),
        'latency_seconds_mean': float(active.loc[TEST, 'seconds'].mean()),
    }
    state = FRAME.iloc[TEST].state.to_numpy()
    for st, nm in enumerate(CELL_NAMES):
        m = state == st
        out[nm + '_delta_EM'] = float(d[m].mean())
        out[nm + '_n'] = int(m.sum())
    return out


FINAL_QA = pd.DataFrame([
    qa_metrics(B, B, 'noop'),
    qa_metrics(B, S, 'E13_sparse_probe_aligned_gate'),
    qa_metrics(B, F, 'full_hidden_probe_gate_DIAGNOSTIC'),
    qa_metrics(B, R, 'sparse_gate_random_direction'),
])
display(FINAL_QA)
FINAL_QA.to_csv(E13_ROOT / 'FINAL_QA_SUMMARY.csv', index=False)


def final_state_metrics(family, mode):
    pk, pc, _ = probs_for_family(family, mode, TEST)
    pred_k = (pk >= 0.5).astype(int)
    pred_c = (pc >= 0.5).astype(int)
    pred_state = cell_ids(pred_k, pred_c)
    true_k = FRAME.iloc[TEST].k.to_numpy(int)
    true_c = FRAME.iloc[TEST].c.to_numpy(int)
    true_state = FRAME.iloc[TEST].state.to_numpy(int)
    return {
        'family': family, 'mode': mode,
        'K_AUROC': float(roc_auc_score(true_k, pk)),
        'K_BAcc': float(balanced_accuracy_score(true_k, pred_k)),
        'C_AUROC': float(roc_auc_score(true_c, pc)),
        'C_BAcc': float(balanced_accuracy_score(true_c, pred_c)),
        'four_BAcc': float(balanced_accuracy_score(true_state, pred_state)),
        'four_macro_F1': float(f1_score(true_state, pred_state, average='macro', zero_division=0)),
        'four_accuracy': float(np.mean(true_state == pred_state)),
        'confusion': confusion_matrix(true_state, pred_state, labels=[0,1,2,3]).tolist(),
    }


STATE_FINAL = {
    'sparse_staged': final_state_metrics('sparse', 'staged'),
    'sparse_late': final_state_metrics('sparse', 'late'),
    'full_staged': final_state_metrics('full', 'staged'),
    'full_late': final_state_metrics('full', 'late'),
}
print(json.dumps(STATE_FINAL, indent=2))
(E13_ROOT / 'FINAL_STATE_METRICS.json').write_text(json.dumps(STATE_FINAL, indent=2), encoding='utf-8')

# E11 reference — E13 selection에는 사용하지 않음
ref = E11_SUMMARY[E11_SUMMARY.arm.eq('utility_staged')]
if len(ref):
    print('\nE11 utility_staged reference')
    display(ref[['baseline_EM','steered_EM','Recovery','Damage','NetGain_count','gate_rate']])

FINAL = {
    'policy_lock_sha256': LOCK_HASH,
    'sparse_policy': SPARSE_POLICY,
    'full_policy_diagnostic': FULL_POLICY,
    'qa': FINAL_QA.to_dict('records'),
    'state': STATE_FINAL,
    'attribution_diagnostics': ATTR_DIAG,
    'axis_compare': AXIS_COMPARE,
}
(E13_ROOT / 'FINAL_RESULT.json').write_text(json.dumps(FINAL, ensure_ascii=False, indent=2), encoding='utf-8')

x = np.arange(len(FINAL_QA))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x, FINAL_QA['NetGain_count'])
ax.axhline(0, linestyle=':')
ax.set_xticks(x, FINAL_QA['method'], rotation=25, ha='right')
ax.set_ylabel('Net Gain count')
fig.tight_layout()
fig.savefig(E13_ROOT / 'final_netgain.png', dpi=160)
plt.show()

print('saved:', E13_ROOT)

# 14. 결과를 읽는 순서

## [효과] 1 — E11 probe axis가 재현되는가?

먼저 `dev_binary_probe_report.csv`와 `axis_compare`를 본다.

- K7-Q / K13-L / C13-L binary probe가 개발셋에서 실제로 분리력을 갖는지
- E11 4-state probe에서 유도한 K/C 축과 binary probe 방향이 얼마나 비슷한지

이 단계가 무너지면 이후 mechanistic attribution 해석을 강하게 주장하면 안 된다.

---

## [효과] 2 — Probe score가 소수 component에 집중되는가?

- `K7_alllayer_head_attribution.csv`
- `K13_alllayer_head_attribution.csv`
- `C13_alllayer_head_attribution.csv`
- `*_alllayer_ffn_top256_perlayer.csv`
- `*_layer_head_attribution_heatmap.png`

특히 `abs_mean_diff`가 특정 `(layer, head)` 또는 FFN neuron에 집중되는지 확인한다.

`decomposition_abs_error_mean`이 충분히 작아야 “probe score를 실제 residual update로 분해했다”는 해석이 성립한다.

---

## [효과] 3 — Sparse score가 full hidden probe를 얼마나 보존하는가?

`sparse_probe_dev_report.csv`

비교:

- full hidden
- probe-aligned sparse
- matched random sparse

**Sparse가 full에 근접하고 random sparse보다 높으면** E11 signal의 상당 부분을 소수 component로 압축할 수 있다는 근거다.

---

## [효과] 4 — Predictive가 아니라 causal한가?

- `causal_prefill_ablation.csv`
- `causal_generation_ablation.csv`
- `paired_patching_summary.csv`

성공 패턴:

1. top attribution component 제거가 random matched보다 full probe score/BAcc를 더 크게 떨어뜨림
2. 실제 generation behavior도 더 많이 변함
3. C+/C− paired patch가 C probe score를 donor 조건 방향으로 이동시킴

이 세 가지가 함께 나오면 단순 linear decodability보다 강한 mechanistic evidence가 된다.

---

## [효과] 5 — Sparse representation이 실제 gate로도 작동하는가?

`FINAL_QA_SUMMARY.csv`

주 비교:

- No-op
- **E13 sparse probe-aligned gate**
- Full hidden probe gate
- Sparse gate + random steering direction

핵심:

- `NetGain_count > 0`
- Damage가 낮음
- random direction보다 실제 instruction direction이 좋음
- full hidden gate와 sparse gate의 성능 격차가 과도하지 않음

---

# 다음 분기

### A. Sparse ≈ Full + causal effect 확인
다음 단계에서 label 의존성을 줄인다.

- label/class = 5, 10, 20, 50, full
- head/FFN component selection stability
- instruction-contrast 기반 unsupervised component ranking
- fold별 component overlap

### B. Sparse probe는 강하지만 ablation/patching이 약함
`decodable but not causal` 가능성이 크다. Gate용 signal과 actuator를 분리해서 설계한다.

### C. Sparse가 full probe보다 크게 떨어짐
K/C representation이 특정 소수 head가 아니라 **distributed subspace**일 가능성이 있다. 다음 후보는 low-rank projection / SAE / distributed sparse subspace다.

### D. Causal component는 찾았는데 QA Net Gain이 0
representation 문제가 아니라 **route / steering direction / strength** 병목일 수 있다. `policy_calibration_sparse.csv`에서 state confidence와 forced-action gain을 분리해서 본다.

---

# 이 실험에서 주장하지 않는 것

- 완전 training-free
- 완전 label-free
- K/C가 단 하나의 head에 저장됨
- attention=외부지식, FFN=내부지식이라는 완전 분해
- single fold 결과의 일반화

이 노트북의 1차 목표는 **E11 probe의 mechanistic localization + causal validation + sparse gate 가능성 검증**이다.